# A Transformer-Based Resume Parser for Multiple File Formats
This notebook showcases how Gemma 7b model can efficiently extract key information from resumes, streamlining the hiring process with its advanced NLP capabilities.

In [1]:
# Install necessary packages for reading docx and pdf
!pip install pymupdf
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 5.6 MB/s eta 0:00:00


In [2]:
import fitz  

# Function to extract text content from a PDF file using pymupdf or fitz
def extract_text_pymupdf(file_path):
    text = ""
    document = fitz.open(file_path)
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        if page_text:
            page_text = page_text.replace('\n', ' ')
            text += page_text
    return text


In [3]:
import docx

# Function to extract text content from a docx file using docx
def read_docx(file_path):
    doc = docx.Document(file_path)
    text = ""
    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"
    return text


In [4]:
import torch                                        
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,  GenerationConfig
import re
import json
import os
import gc

In [5]:
# Load Gemma 7b model and tokenizer for resume parsing
model_name = "/kaggle/input/gemma/transformers/1.1-7b-it/1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             torch_dtype=torch.bfloat16, 
                                             device_map="auto")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
torch.backends.cuda.enable_mem_efficient_sdp(False)

In [7]:
model.device

device(type='cuda', index=0)

In [8]:
model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 3072, padding_idx=0)
    (layers): ModuleList(
      (0-27): 28 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (v_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3072, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (up_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (down_proj): Linear(in_features=24576, out_features=3072, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
      )
    )
    (norm): Gemm

In [9]:
# Clear CUDA cache and perform garbage collection to free up GPU memory
torch.cuda.empty_cache()
gc.collect()

21

In [10]:
# Function to extract key information from a resume text and output it in JSON format
# The function takes resume_text as input and optionally a file name for the output
def model_output(resume_text, file='output'):
    # Construct a prompt with desired information to extract
    prompt = f"""
    Extract key information such as:
     - Contact Information (Name, Email, Phone Number)
     - Education History (Institution, Degree, Graduation Year)
     - Work Experience (Company, Position, Duration, Description)
     - Skills
     - Additional sections as applicable (Certifications, Projects, etc.)
     Ouput in JSON format, so I can integrate it directly
    <Resume>
    {resume_text}
    </End of Resume>
     """
    
    # Initialize chat with the prompt
    chat = [
                { "role": "user", "content": prompt }
            ]
    
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer.encode(text, return_tensors="pt")
    
    output = model.generate(input_ids=input_ids.to(model.device), num_return_sequences=1, max_new_tokens=5000)
    decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
    
    try:
        # Extract JSON output and save it to a file
        output = re.findall(pattern="model.*```json(.*)```", string=decoded_output, flags=re.S)[0]
        output = output.strip()

        s = json.loads(output)
        with open(file+'.json', 'w') as f:
            json.dump(s,f,indent=4)
            
        return True
    except Exception as e:
        print("ERROR",e)
    
    # If an error occurs, include error message in the prompt and generate output again
    chat = [
                { "role": "user", "content": output + "\n\n" + str(e) + "\nFix the error and give complete output so I can integrate it directly"}
            ]
    
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer.encode(text, return_tensors="pt")
    output = model.generate(input_ids=input_ids.to(model.device), num_return_sequences=1, max_new_tokens=5000)
    decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract JSON output and save it to a file
    output = re.findall(pattern="model.*```json(.*)```", string=decoded_output, flags=re.S)[0]
    output = output.strip()
        
    s = json.loads(output)
    with open(file+'.json', 'w') as f:
        json.dump(s,f,indent=4)
        
    return True



In [11]:
directory = '/kaggle/input/sample-resumes/Sample Resumes/'
# Loop through files in the directory
for file_name in os.listdir(directory):
    file_path = os.path.join(directory, file_name)
    
    # Extract text based on file type
    if file_name.endswith('.pdf'):
        resume_text = extract_text_pymupdf(file_path)
        file_name = file_name[:-4]
    elif file_name.endswith('.docx'):
        resume_text = read_docx(file_path)
        file_name = file_name[:-5]
    else:
        print("Unsupported file type:", file_name)    
        continue
    
    # Process the resume text using the model_output function
    try:
        model_output(resume_text, file_name)
    except Exception as e:
        print("Error processing file", file_name + ":", e)
    
    # Clear CUDA cache and perform garbage collection
    torch.cuda.empty_cache()
    gc.collect()


2024-05-26 17:04:50.407295: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-26 17:04:50.407437: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-26 17:04:50.543971: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
